# BirdCLEF 2026 — CPU Inference Notebook

**Competition constraints**: CPU only (GPU = 1 min limit), no internet, 90 min budget.

**Before running — attach two extra datasets:**
1. Your **code dataset** (contains the `src/` folder) → set `CODE_DATASET_SLUG`
2. Your **weights dataset** (contains `best.pt`) → set `WEIGHTS_DATASET_SLUG`

Both are uploaded to Kaggle → Datasets → New Dataset.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
CODE_DATASET_SLUG    = "your-code-dataset-slug"      # e.g. youssef/birdclef-src
WEIGHTS_DATASET_SLUG = "your-weights-dataset-slug"   # e.g. youssef/birdclef-weights
CHECKPOINT_FILENAME  = "best.pt"
MODEL_NAME           = "efficientnet_b3"              # must match trained model
BATCH_SIZE           = 32                             # windows per CPU batch
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, time
CODE_DIR    = f"/kaggle/input/{CODE_DATASET_SLUG}"
WEIGHTS_DIR = f"/kaggle/input/{WEIGHTS_DATASET_SLUG}"
COMP_DIR    = "/kaggle/input/birdclef-2026"
OUT_CSV     = "/kaggle/working/submission.csv"

for label, path in [("Code dir", CODE_DIR), ("Weights dir", WEIGHTS_DIR), ("Comp dir", COMP_DIR)]:
    ok = os.path.isdir(path)
    print(f"  {label}: {'OK' if ok else 'MISSING — check your slug'}  ({path})")

In [ ]:
# Add src/ to path — must point to the src/ subdirectory, not the root
SRC_DIR = os.path.join(CODE_DIR, "src")
assert os.path.isdir(SRC_DIR), f"src/ not found at {SRC_DIR}"
sys.path.insert(0, SRC_DIR)

from config import Config, ALL_CLASSES, NUM_CLASSES
print(f"Config loaded — {NUM_CLASSES} classes")

In [ ]:
# Override all data paths to competition directory
from pathlib import Path

cfg = Config()
cfg.train_audio_dir       = Path(COMP_DIR) / "train_audio"
cfg.train_soundscapes_dir = Path(COMP_DIR) / "train_soundscapes"
cfg.test_soundscapes_dir  = Path(COMP_DIR) / "test_soundscapes"
cfg.train_csv             = Path(COMP_DIR) / "train.csv"
cfg.taxonomy_csv          = Path(COMP_DIR) / "taxonomy.csv"
cfg.soundscape_labels_csv = Path(COMP_DIR) / "train_soundscapes_labels.csv"
cfg.sample_submission_csv = Path(COMP_DIR) / "sample_submission.csv"

test_files = sorted(cfg.test_soundscapes_dir.glob("*.ogg"))
print(f"Test soundscapes: {len(test_files)}")
if test_files:
    print(f"  First: {test_files[0].name}")
    print(f"  Last:  {test_files[-1].name}")

In [ ]:
# Runtime estimate before we start
import math

# Estimate windows: load one file to check its length
sample_file = test_files[0] if test_files else None
if sample_file:
    try:
        import soundfile as sf
        info = sf.info(str(sample_file))
        file_dur_s = info.duration
    except Exception:
        import librosa
        file_dur_s = librosa.get_duration(path=str(sample_file))
    windows_per_file = math.floor(file_dur_s / cfg.clip_duration)
    total_windows    = len(test_files) * windows_per_file
    total_batches    = math.ceil(total_windows / BATCH_SIZE)

    # EfficientNet-B3 on CPU: ~80ms/batch of 32 (varies by machine)
    est_infer_s  = total_batches * 0.08
    # Audio loading: ~0.15s/file
    est_load_s   = len(test_files) * 0.15
    est_total_min = (est_infer_s + est_load_s) / 60

    print(f"File duration:    {file_dur_s:.0f}s  →  {windows_per_file} windows/file")
    print(f"Total windows:    {total_windows:,}")
    print(f"Total batches:    {total_batches:,}  (batch_size={BATCH_SIZE})")
    print(f"Estimated time:   ~{est_total_min:.1f} min  (budget: 90 min)")
    if est_total_min > 75:
        print("⚠  Close to budget — consider increasing BATCH_SIZE or switching to efficientnet_b0")

In [ ]:
# Load model — works with timm OR torchvision (no internet needed)
import torch
from models import build_model

device = torch.device("cpu")   # GPU disabled for Kaggle code submissions
print(f"Device: {device}")

ckpt_path = os.path.join(WEIGHTS_DIR, CHECKPOINT_FILENAME)
print(f"Loading: {ckpt_path}")

ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
model = build_model(MODEL_NAME, NUM_CLASSES, pretrained=False)
model.load_state_dict(ckpt["model"])
model.eval()

# Use all CPU cores for intra-op parallelism
torch.set_num_threads(os.cpu_count() or 4)
print(f"Threads: {torch.get_num_threads()}")
print(f"Model loaded — best cmAP={ckpt.get('best_cmap', 0):.4f}  epoch={ckpt.get('epoch', '?')}")

In [ ]:
# Batched inference — all windows from a file are stacked into one batch
from audio_utils import build_mel_transform, soundscape_windows
import pandas as pd

mel      = build_mel_transform(cfg)
all_rows = []
t_start  = time.time()

with torch.no_grad():
    for file_idx, path in enumerate(test_files):

        # --- load all windows for this file at once ---
        windows = soundscape_windows(path, cfg, mel)  # list of (end_sec, tensor)
        if not windows:
            continue

        end_secs  = [w[0] for w in windows]
        specs     = torch.stack([w[1] for w in windows])  # (W, 1, mels, frames)

        # --- batched forward pass ---
        probs_list = []
        for i in range(0, len(specs), BATCH_SIZE):
            batch  = specs[i : i + BATCH_SIZE]          # (B, 1, mels, frames)
            logits = model(batch)                        # (B, 234)
            probs_list.append(torch.sigmoid(logits))
        probs_all = torch.cat(probs_list, dim=0).numpy()  # (W, 234)

        # --- build rows ---
        stem = path.stem
        for end_sec, probs in zip(end_secs, probs_all):
            row = {"row_id": f"{stem}_{end_sec}"}
            row.update(zip(ALL_CLASSES, probs.tolist()))
            all_rows.append(row)

        # --- progress every 50 files ---
        if (file_idx + 1) % 50 == 0 or (file_idx + 1) == len(test_files):
            elapsed  = time.time() - t_start
            per_file = elapsed / (file_idx + 1)
            remaining = per_file * (len(test_files) - file_idx - 1)
            print(f"  {file_idx+1}/{len(test_files)} files | "
                  f"{len(all_rows)} windows | "
                  f"elapsed {elapsed/60:.1f}m | "
                  f"ETA {remaining/60:.1f}m")

print(f"\nInference complete — {len(all_rows)} total windows in {(time.time()-t_start)/60:.1f} min")

In [ ]:
# Save submission.csv
submission = pd.DataFrame(all_rows, columns=["row_id"] + ALL_CLASSES)
submission.to_csv(OUT_CSV, index=False)
print(f"Saved → {OUT_CSV}")
print(f"Shape: {submission.shape}")

# Validate against sample_submission
sample = pd.read_csv(cfg.sample_submission_csv)
ok = list(submission.columns) == list(sample.columns)
print(f"Columns match sample_submission.csv: {ok}")
if not ok:
    missing = set(sample.columns) - set(submission.columns)
    extra   = set(submission.columns) - set(sample.columns)
    if missing: print(f"  Missing: {missing}")
    if extra:   print(f"  Extra:   {extra}")

# Preview
print("\nFirst 3 rows:")
print(submission[["row_id"]].head(3).to_string(index=False))